# Qwen-3.5-2B benchmarking.
- Dataset: MME-RealWorld
- Inference: vLLM
- Model Weights: HuggingFace

### Setup

Need to connect to `Google Drive` to avoid installing dependencies and dataset/weights each run, connect to `HuggingFace` in order to pull model weights, initialize environment variables.

In [1]:
# Choose whichever should be tested.
BENCHMARK_MODELS = {
    "qwen3.5-2b": True,
    "qwen3.5-2b-awq": True,
    "qwen3.5-2b-gptq": False
}

In [2]:
#* connect to google drive (storage)
from google.colab import drive

# aim: store model weights and dataset in drive
# force_remount ensures retries in case of a connection error
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
#* set up environment variables
#! Should be done at the beginning.
import os

# Google Drive directory, where everything is going to be stored
PROJECT_DIR = os.environ["PROJECT_DIR"] = "/content/drive/MyDrive/qwen3.5-quant-bench"
# subdirectory for huggingface cache
HF_HOME = os.environ["HF_HOME"] = os.path.join(PROJECT_DIR, "hf_cache")

# dataset repo id
DS_REPO_ID = "yifanzhang114/MME-RealWorld"

# Repo id + vLLM `quantization` kwarg per benchmarked variant.
# quantization=None means full precision (fp16).
MODEL_CONFIGS = {
    "qwen3.5-2b": {
        "repo_id": "Qwen/Qwen3.5-2B",
        "quantization": None,
    },
    "qwen3.5-2b-awq": {
        "repo_id": "QuantTrio/Qwen3.5-2B-AWQ",
        "quantization": "awq",
    },
    "qwen3.5-2b-gptq": {
        # TODO: as of 2026-09 there is no public Qwen3.5-2B GPTQ checkpoint on
        # HF (checked QuantTrio and the wider Hub). Either fill this in once
        # one ships, quantize your own with GPTQModel/AutoGPTQ and point this
        # at the local/HF path, or set BENCHMARK_MODELS["qwen3.5-2b-gptq"] =
        # False. Left enabled-but-empty on purpose so the skip is loud below
        # instead of silently missing from the benchmark.
        "repo_id": "",
        "quantization": "gptq",
    },
}

# retrieve huggingface token from Colab secrets.
#
# userdata.get() needs a round-trip to the actual Colab browser tab to show
# its permission prompt, so it only works when THIS cell is run from that tab
# -- it times out otherwise (e.g. if driven from VSCode's Jupyter extension
# connected to this same kernel). Deliberately not persisting the token to a
# file on Drive as a workaround: a file there is readable by anything with
# Drive access to this folder (any other notebook, any app you've granted
# Drive access, Drive Desktop sync, anyone this folder gets shared with), not
# just this notebook, unlike the vault. If you're driving this notebook from
# VSCode: run this one cell from the actual Colab UI once per fresh runtime
# (the token then lives only in this kernel's memory for the rest of the
# session), then switch back to VSCode for everything else.
# if os.getenv("HF_TOKEN") is None:
#     try:
#         from google.colab import userdata
#         os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_API_KEY"] = userdata.get("HF_API_KEY")
#     except TimeoutError:
#         print("Could not retrieve HF Access Token. Timed out.")
# Rust-based accelerated downloader for HF Hub transfers (needs the hf_transfer
# package, installed below) -- meaningfully faster than the default downloader
# for large weight files.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [5]:
%%bash
pip install uv # faster than python venv
uv venv --python 3.12 --seed
source .venv/bin/activate
uv pip install vllm --torch-backend=auto
# vLLM raises an error if torchaudio is installed and CUDA version is different
# But I don't need torchaudio anyway.
uv pip uninstall torchaudio --system
# [hf_transfer] extra pulls in the Rust-based accelerated downloader used by
# HF_HUB_ENABLE_HF_TRANSFER above.
uv pip install "huggingface_hub[hf_transfer]"
# Dataset evaluation code. Guarded so re-running this cell mid-session doesn't
# error on an already-cloned directory (this lives under /content, not Drive,
# so it does need to be re-cloned every fresh Colab runtime).
if [ ! -d /content/MME-RealWorld ]; then
  git clone https://github.com/MME-Benchmarks/MME-RealWorld.git /content/MME-RealWorld
fi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 62.4 MB/s eta 0:00:00


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment with seed packages at: .venv
 + pip==26.2.1
Activate with: source .venv/bin/activate
Using Python 3.13.15 environment at: /usr
Resolved 183 packages in 2.03s
Prepared 88 packages in 55.31s
Uninstalled 8 packages in 793ms
Installed 88 packages in 1.19s
 + agent-detector==2.0.0
 + anthropic==1.6.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 + cuda-bindings==13.4.1
 + cuda-core==1.2.0
 + cuda-pathfinder==1.8.1
 + cuda-python==13.4.1
 + cuda-tile==1.6.0
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + fastsafetensors==0.4.0
 + flashinfer-python==0.6.18
 + humming-kernels==0.1.12
 + ijson==3.5.1
 + instanttensor==0.2.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + ll

In [6]:
# set up hugging face api
import getpass, os

# take api token from secret input, write to drive once, then reuse it
# it's not an issue since token access rights are restricted to public repos
ENV_FILE = "/content/.qwenbench.env"  # ephemeral VM disk, deliberately NOT under PROJECT_DIR/Drive
if not os.path.exists(ENV_FILE):
    token = getpass.getpass("HF token (input hidden): ")
    with open(ENV_FILE, "w") as f:
        f.write(f"HF_TOKEN={token}\n")

from dotenv import load_dotenv
load_dotenv(ENV_FILE)
os.environ["HUGGINGFACE_API_KEY"] = os.environ["HF_TOKEN"]

In [7]:
#* will be used later to halt unnecessary cell execution.
class StopCellExecution(Exception):
    def _render_traceback_(self): # Hides the ugly error traceback output
        return []

In [8]:
#* download model weights
from concurrent.futures import ThreadPoolExecutor

from huggingface_hub import snapshot_download

def _download_model_weights(key: str, repo_id: str) -> tuple[str, str]:
    return key, snapshot_download(repo_id=repo_id, cache_dir=HF_HOME)

to_download = {
    key: cfg["repo_id"]
    for key, cfg in MODEL_CONFIGS.items()
    if BENCHMARK_MODELS.get(key) and cfg["repo_id"]
}
for key in BENCHMARK_MODELS:
    if BENCHMARK_MODELS[key] and key not in to_download:
        print(f"Skipping {key}: no repo_id configured in MODEL_CONFIGS.")

# Every model's weights are an independent HF download -- fetch them
# concurrently instead of waiting on each snapshot_download in turn.
with ThreadPoolExecutor(max_workers=max(1, len(to_download))) as pool:
    for key, weights_path in pool.map(lambda kv: _download_model_weights(*kv), to_download.items()):
        MODEL_CONFIGS[key]["weights_path"] = weights_path
        print(f"Finished downloading {key}: {weights_path}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Finished downloading qwen3.5-2b: /content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
Finished downloading qwen3.5-2b-awq: /content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/models--QuantTrio--Qwen3.5-2B-AWQ/snapshots/b9b439ecbe34b097cfc9531132fedbf55fec9263


In [9]:
#* download dataset
from huggingface_hub import snapshot_download

# download dataset
dataset_path = snapshot_download(
    repo_id=DS_REPO_ID,
    repo_type="dataset",
    cache_dir=HF_HOME
)

SNAPSHOT_ROOT = dataset_path

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

In [10]:
#* check if there's any broken archive
import glob

def check_all_targets(snapshot_root: str) -> list[str]:
    broken = []
    for f in glob.glob(os.path.join(snapshot_root, "*")):
        if os.path.islink(f) and not os.path.exists(f):
            broken.append(f)
    return broken

broken = check_all_targets(SNAPSHOT_ROOT)
for b in broken:
    print("BROKEN:", os.path.basename(b))
print(f"\n{len(broken)} broken symlinks found out of total files")


0 broken symlinks found out of total files


In [11]:
#* download broken archives again
from huggingface_hub import hf_hub_download

for b in broken:
    filename = os.path.basename(b)
    print(f"Re-downloading {filename}...")
    hf_hub_download(
        repo_id=DS_REPO_ID,
        repo_type="dataset",
        filename=filename,
        cache_dir=HF_HOME,
        force_download=True,
    )

In [12]:
if check_all_targets(SNAPSHOT_ROOT):
    raise RuntimeError("Broken symlinks still exist")

In [13]:
import subprocess, glob, os

# extract archived images
EXTRACT_DIR = os.path.join(PROJECT_DIR, "mme_dataset_images")
os.makedirs(EXTRACT_DIR, exist_ok=True)

part_files = glob.glob(os.path.join(SNAPSHOT_ROOT, "*.tar.gz.part_*"))
plain_files = glob.glob(os.path.join(SNAPSHOT_ROOT, "*.tar.gz"))

# base name -> path, built so plain files always win over parts when both exist
plain_by_base = {f[:-len(".tar.gz")]: f for f in plain_files}
# use 'set' because multiple part files will have the same base name
part_bases = sorted(set(f.rsplit(".tar.gz.part_", 1)[0] for f in part_files))

to_extract = []  # list of (archive_path, needs_reassembly: bool, parts: list)
handled_bases = set()

# first look at parted archives
for base in part_bases:
    plain = plain_by_base.get(base)
    # if there's a plain archive, ignore parts, use plain
    if plain and os.path.exists(plain): # follows symlink; False if broken
        print(f"Using existing {os.path.basename(plain)} (skipping .part_ files for this base)")
        to_extract.append((plain, False, None))
    # otherwise, add parts to be concatenated
    else:
        parts = sorted(glob.glob(base + ".tar.gz.part_*"))
        archive_name = base + ".tar.gz"
        to_extract.append((archive_name, True, parts))
    handled_bases.add(base)

# look for plain bases that don't have part counterparts
# , add them to be processed
for base, plain in plain_by_base.items():
    if base not in handled_bases:
        to_extract.append((plain, False, None))

# reassemble where needed, then extract everything
for archive_name, needs_reassembly, parts in to_extract:
    if os.path.exists(archive_name):
      continue
    # assemble parts if needed
    if needs_reassembly:
        print(f"Reassembling {os.path.basename(archive_name)} from {len(parts)} parts...")
        with open(archive_name, "wb") as out_f:
            for p in parts:
                with open(p, "rb") as in_f:
                    out_f.write(in_f.read())
    # extract data
    print(f"Extracting {os.path.basename(archive_name)}...")
    subprocess.run(["tar", "-xzf", archive_name, "-C", EXTRACT_DIR], check=True)

print("Done. Extracted contents:", os.listdir(EXTRACT_DIR))

Using existing MME-HD-CN.tar.gz (skipping .part_ files for this base)
Using existing diagram_and_table.tar.gz (skipping .part_ files for this base)
Using existing monitoring_images.tar.gz (skipping .part_ files for this base)
Using existing ocr_cc.tar.gz (skipping .part_ files for this base)
Using existing remote_sensing.tar.gz (skipping .part_ files for this base)
Done. Extracted contents: ['dota_v2_dota_v2_dota_v2_P11113.png', 'dota_v2_dota_v2_dota_v2_P11114.png', 'dota_v2_dota_v2_dota_v2_P11115.png', 'dota_v2_dota_v2_dota_v2_P11116.png', 'dota_v2_dota_v2_dota_v2_P11117.png', 'dota_v2_dota_v2_dota_v2_P11118.png', 'dota_v2_dota_v2_dota_v2_P11119.png', 'dota_v2_dota_v2_dota_v2_P11120.png', 'dota_v2_dota_v2_dota_v2_P11121.png', 'dota_v2_dota_v2_dota_v2_P11124.png', 'dota_v2_dota_v2_dota_v2_P11126.png', 'dota_v2_dota_v2_dota_v2_P11123.png', 'dota_v2_dota_v2_dota_v2_P11125.png', 'dota_v2_dota_v2_dota_v2_P11127.png', 'dota_v2_dota_v2_dota_v2_P11122.png', 'dota_v2_dota_v2_dota_v2_P11129.png

In [14]:
import os
from collections import deque

def listsubdirs(_dir: str) -> list[str]:
  return [os.path.join(_dir, p) for p in os.listdir(_dir)]

# put all images into a single directory, because database's directory structure
# is a nightmare and it doesn't affect results (everything is tied to questions)
q = deque()
q.append(EXTRACT_DIR)
while q:
  p = q.pop()
  if os.path.isdir(p):
    for d in listsubdirs(p):
      q.append(d)
  elif not p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
    continue
  else:
    os.rename(p, os.path.join(EXTRACT_DIR, os.path.basename(p)))

In [15]:
from PIL import Image
import time

#! use this instead of PIL.Image.open as google drive might not have indexed that file yet
#! See: Drive FUSE mount issue
def safe_open_drive_img(image_path: str, retries: int=3, delay: float=1.0, from_basename: bool = True) -> Image:
  if from_basename:
    image_path = os.path.join(EXTRACT_DIR, os.path.basename(image_path))
  for attempt in range(retries):
      try:
          return Image.open(image_path).convert("RGB")
      except FileNotFoundError:
          if attempt == retries - 1:
              raise
          time.sleep(delay)

In [16]:
import os, glob

# locate question list json
json_candidates = glob.glob(os.path.join(SNAPSHOT_ROOT, "**", "*.json"), recursive=True)
assert json_candidates, "No JSON file found in snapshot — inspect the printed listing above."

print(f"Discovered potential JSON files: {json_candidates}")

# ignore chinese and look for english
QUESTIONS_FILE = [candidate for candidate in json_candidates if "MME" in os.path.basename(candidate) and not "CN" in candidate][0]

print("Using questions file:", QUESTIONS_FILE)

Discovered potential JSON files: ['/content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/datasets--yifanzhang114--MME-RealWorld/snapshots/741cb8831ac86085bd54f678d13ca193e2334114/MME_RealWorld_CN.json', '/content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/datasets--yifanzhang114--MME-RealWorld/snapshots/741cb8831ac86085bd54f678d13ca193e2334114/MME_RealWorld.json']
Using questions file: /content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/datasets--yifanzhang114--MME-RealWorld/snapshots/741cb8831ac86085bd54f678d13ca193e2334114/MME_RealWorld.json


In [17]:
import json

# load questions
with open(QUESTIONS_FILE, "r") as f:
    questions = json.load(f)
print(f"Loaded {len(questions)} questions")

# kept distinct from `questions` -- the sampling cell below reassigns
# `questions` to a subset, and needs the full set to sample from.
all_questions = questions

PROMPT_SUFFIX = (
    "Select the best answer to the above multiple-choice question based on the image. "
    "Respond with only the letter (A, B, C, D, or E) of the correct option.\n"
    "The best answer is:"
)

from typing import Dict

def build_chat(item: Dict) -> Dict:
  """Take question item from dataset and turn into prompt list."""
  choices_text = "The choices are listed below:\n" + "\n".join(item["Answer choices"])
  text = f"{item['Text']}\n{choices_text}\n{PROMPT_SUFFIX}"
  return [
      {
          "role": "user",
          "content": [
              # safe_open_drive_img already returns an RGB image -- don't
              # convert() a second time, it just recopies the buffer.
              {"type": "image_pil", "image_pil": safe_open_drive_img(item["Image"])},
              {"type": "text", "text": text},
          ],
      }
  ]

Loaded 23609 questions


## Inference

In [ ]:
# construct sample dataset
# take 10% of each task
import os, json

SAMPLE_QUESTIONS_FILE = os.path.join(PROJECT_DIR, "sample_questions.json")

if os.path.exists(SAMPLE_QUESTIONS_FILE):
  with open(SAMPLE_QUESTIONS_FILE, "r") as f:
    questions = json.load(f)
else:
  import random

  DS_COMPR_RATE = 0.1

  # Sample from `all_questions` (the full set loaded above), not `questions` --
  # `questions` is what this block produces (the sampled subset), so it can't
  # also be the source to sample from.
  reas = [q for q in all_questions if q.get('Task') == 'Reasoning']
  perc = [q for q in all_questions if q.get('Task') == 'Perception']

  reas = random.sample(reas, int(DS_COMPR_RATE * len(reas)))
  perc = random.sample(perc, int(DS_COMPR_RATE * len(perc)))

  questions = reas + perc

  with open(SAMPLE_QUESTIONS_FILE, "w") as f:
    json.dump(questions, f, indent=2)

print(f"Using {len(questions)} sampled questions")

Using 2360 sampled questions


: 

In [ ]:
#* stage the sampled images locally, once
import shutil
from concurrent.futures import ThreadPoolExecutor

# Reading straight off Drive means one Drive FUSE network round trip per
# image, looked up inside EXTRACT_DIR -- a single flat directory holding
# every image in the full dataset, not just the sample. Lookups against a
# Drive-mounted directory that large are slow on their own, and Drive's
# per-user rate limiting can add further silent backoff under concurrent
# access on top of that. A one-time bulk copy of just the sampled images to
# local disk turns that into local reads for the actual decode step below,
# which are effectively free by comparison. Skips files already staged, so
# an interrupted run resumes instead of starting the copy over.
LOCAL_IMG_DIR = "/content/mme_sample_images"
os.makedirs(LOCAL_IMG_DIR, exist_ok=True)

basenames = sorted({os.path.basename(q["Image"]) for q in questions})
to_stage = [b for b in basenames if not os.path.exists(os.path.join(LOCAL_IMG_DIR, b))]

def _stage_one(basename: str) -> None:
    shutil.copyfile(
        os.path.join(EXTRACT_DIR, basename),
        os.path.join(LOCAL_IMG_DIR, basename),
    )

# Modest worker count on purpose: this step still hits Drive, so more threads
# isn't obviously better against a rate-limited remote API the way it is for
# purely local work.
with ThreadPoolExecutor(max_workers=8) as pool:
    done = 0
    for _ in pool.map(_stage_one, to_stage):
        done += 1
        if done % 100 == 0 or done == len(to_stage):
            print(f"Staged {done}/{len(to_stage)} images")

print(f"{len(basenames) - len(to_stage)} already staged from a prior run, {len(to_stage)} copied this run.")

In [ ]:
# Pre-build every chat payload once. The images/questions are identical across
# every quantization method being benchmarked below, so decoding each image
# and building its chat payload separately per model (3x for fp16/awq/gptq)
# would be pure repeated decode work for the same bytes.

#! Trade-off: this holds every decoded image in memory for the whole run, which
#! is fine for the 10% sample but should be reconsidered (e.g. build chats
#! per-chunk instead) for a full-dataset run.
from concurrent.futures import ThreadPoolExecutor

# safe_open_drive_img resolves images relative to the global EXTRACT_DIR --
# point it at the local staged copy (see the cell above) for this step, then
# restore it, so other cells that expect EXTRACT_DIR to mean the Drive copy
# aren't affected by the swap.
_drive_extract_dir = EXTRACT_DIR
EXTRACT_DIR = LOCAL_IMG_DIR
try:
    with ThreadPoolExecutor(max_workers=16) as pool:
        chats = list(pool.map(build_chat, questions))
finally:
    EXTRACT_DIR = _drive_extract_dir

print(f"Pre-built {len(chats)} chat payloads.")

#* Print the size of the pre-built chats.
import sys
print(f"Pre-built chats take up {sys.getsizeof(chats) / 1024 :.2f} Kb in total.")

In [ ]:
import gc
import json
import time

import numpy as np
import torch
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel

# Chunk size doubles as vLLM's concurrent scheduling window (max_num_seqs) and
# the incremental-checkpoint boundary, so each chunk is sized to actually
# saturate the scheduler (continuous batching), not an arbitrary number
# unrelated to how many requests vLLM can run at once.
MAX_NUM_SEQS = 64
BATCH_SIZE = MAX_NUM_SEQS
# Batch sizes vLLM pre-captures a CUDA graph for. Covers the partial last chunk too.
CUDAGRAPH_SIZES = [1, 2, 4, 8, 16, 32, 64]


def _percentiles(values: list[float]) -> dict:
    if not values:
        return {"p50": None, "p90": None, "p99": None}
    arr = np.array(values)
    return {
        "p50": float(np.percentile(arr, 50)),
        "p90": float(np.percentile(arr, 90)),
        "p99": float(np.percentile(arr, 99)),
    }


def _request_metrics(output) -> dict | None:
    """Per-request timing off vLLM's own RequestMetrics, not hand-rolled
    wall-clock timing around the batch call -- batch-level timing hides
    TTFT/TPOT and under-counts latency variance."""
    m = output.metrics
    if m is None or m.first_token_time is None or m.finished_time is None:
        return None
    ntoks = len(output.outputs[0].token_ids)
    return {
        "latency": m.finished_time - m.arrival_time,
        "ttft": m.first_token_time - m.arrival_time,
        "tpot": (
            (m.finished_time - m.first_token_time) / (ntoks - 1) if ntoks > 1 else None
        ),
        "num_tokens": ntoks,
    }


def run_inference_pass(
    model_weights_cache: str,
    results_json: str,
    questions: list[dict],
    chats: list,
    quantization: str | None = None,
) -> dict:
    llm_kwargs = dict(
        model=model_weights_cache,
        # T4 (this notebook's Colab GPU) has no native bf16 tensor cores --
        # float16 is the correct dtype here, not just a default choice.
        dtype="float16",
        # Kept the same across every quantization method under test on purpose:
        # letting a smaller quantized model claim more KV-cache headroom would
        # improve its throughput for a reason unrelated to quantization itself,
        # which would make the comparison unfair.
        gpu_memory_utilization=0.85,
        # MME prompts are one image + a short question -- nowhere near vLLM's
        # reported max. Kept deliberately small: a needlessly large
        # max_model_len reserves KV-cache block space for sequence lengths
        # that never happen, directly shrinking how many requests can run
        # concurrently.
        max_model_len=8192,
        max_num_seqs=MAX_NUM_SEQS,
        trust_remote_code=True,
        # Shares the common chat-template preamble across requests; the actual
        # image+question content still differs per request so this is a modest
        # win, not a free lunch, but it costs nothing to enable.
        enable_prefix_caching=True,
        compilation_config={"cudagraph_capture_sizes": CUDAGRAPH_SIZES},
    )
    if quantization:
        llm_kwargs["quantization"] = quantization
        # NOTE: vLLM auto-upgrades "awq"/"gptq" to the Marlin-kernel backend on
        # Ampere+ GPUs. T4 is Turing (sm_75), which Marlin does not support, so
        # expect vLLM to fall back to a slower (but correct) kernel here --
        # check the engine startup log for which kernel actually got selected,
        # since that dominates quantized throughput far more than anything else
        # tunable in this cell.

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    mem_before = torch.cuda.memory_allocated()

    llm = LLM(**llm_kwargs)
    sampling_params = SamplingParams(temperature=0.0, max_tokens=512)

    results = []
    request_metrics = []
    wall_start = time.perf_counter()
    for i in range(0, len(questions), BATCH_SIZE):
        batch_items = questions[i : i + BATCH_SIZE]
        batch_chats = chats[i : i + BATCH_SIZE]
        # one llm.chat() call per chunk -- vLLM continuously batches every
        # request handed to it together, rather than us serializing on
        # chunk boundaries smaller than the scheduler could actually run.
        outputs = llm.chat(batch_chats, sampling_params)
        for item, output in zip(batch_items, outputs):
            item_copy = dict(item)
            item_copy["Output"] = output.outputs[0].text.strip()
            results.append(item_copy)
            rm = _request_metrics(output)
            if rm is not None:
                request_metrics.append(rm)

        # write incrementally in case kernel dies
        with open(results_json, "w") as f:
            json.dump(results, f)

        print(f"Processed {i + len(batch_items)}/{len(questions)}")
    wall_seconds = time.perf_counter() - wall_start

    peak_mem = torch.cuda.max_memory_allocated()

    # clean up before the next model loads
    destroy_model_parallel()
    del llm
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    mem_after_unload = torch.cuda.memory_allocated()

    latencies = [r["latency"] for r in request_metrics]
    ttfts = [r["ttft"] for r in request_metrics]
    tpots = [r["tpot"] for r in request_metrics if r["tpot"] is not None]
    ntoks = [r["num_tokens"] for r in request_metrics]

    time_weighted_latency = (
        sum(l * n for l, n in zip(latencies, ntoks)) / sum(ntoks)
        if latencies and sum(ntoks) > 0
        else None
    )

    metrics = {
        "num_requests": len(results),
        "wall_clock_seconds": wall_seconds,
        "latency": {
            "avg": float(np.mean(latencies)) if latencies else None,
            "time_weighted_avg": time_weighted_latency,
            **_percentiles(latencies),
        },
        "ttft": {
            "avg": float(np.mean(ttfts)) if ttfts else None,
            **_percentiles(ttfts),
        },
        "tpot": {
            "avg": float(np.mean(tpots)) if tpots else None,
            **_percentiles(tpots),
        },
        "memory_gb": {
            "peak_allocated": peak_mem / 1e9,
            "unload_residual": (mem_after_unload - mem_before) / 1e9,
        },
    }
    return {"results": results, "metrics": metrics}

In [ ]:
import json
import os

RESULTS_DIR = os.path.join(PROJECT_DIR, "test_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

all_metrics = {}
for key, cfg in MODEL_CONFIGS.items():
    if not BENCHMARK_MODELS.get(key):
        print(f"Skipping {key}: disabled in BENCHMARK_MODELS.")
        continue
    if "weights_path" not in cfg:
        print(f"Skipping {key}: weights were not downloaded (missing repo_id?).")
        continue

    print(f"\n=== Running inference: {key} ===")
    results_json = os.path.join(RESULTS_DIR, f"{key}_MME_res.json")
    run_output = run_inference_pass(
        cfg["weights_path"], results_json, questions, chats, quantization=cfg["quantization"]
    )
    all_metrics[key] = run_output["metrics"]

    with open(os.path.join(RESULTS_DIR, f"{key}_metrics.json"), "w") as f:
        json.dump(run_output["metrics"], f, indent=2)

with open(os.path.join(RESULTS_DIR, "benchmark_summary.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

In [ ]:
import subprocess

# cloned git repository, not dataset cache
EVAL_SCRIPT = "/content/MME-RealWorld/evaluation/eval_your_results.py"

for key in all_metrics:  # only models that actually ran inference above
    results_json = os.path.join(RESULTS_DIR, f"{key}_MME_res.json")
    eval_txt = os.path.join(RESULTS_DIR, f"{key}_eval_results.txt")
    print(f"Evaluating {key}...")
    with open(eval_txt, "w") as f:
        subprocess.run(
            ["python", EVAL_SCRIPT, "--results_file", results_json],
            stdout=f, stderr=subprocess.STDOUT, check=True,
        )

In [ ]:
def _fmt(x, prec=3):
    return "n/a" if x is None else f"{x:.{prec}f}"

header = f"{'model':<20}{'lat avg':>10}{'lat p99':>10}{'ttft avg':>10}{'tpot avg':>10}{'peak GB':>10}"
print(header)
print("-" * len(header))
for key, m in all_metrics.items():
    print(
        f"{key:<20}"
        f"{_fmt(m['latency']['avg']):>10}"
        f"{_fmt(m['latency']['p99']):>10}"
        f"{_fmt(m['ttft']['avg']):>10}"
        f"{_fmt(m['tpot']['avg']):>10}"
        f"{_fmt(m['memory_gb']['peak_allocated'], 2):>10}"
    )

print(f"\nFull per-run metrics (incl. p50/p90/p99, time-weighted latency, unload residual): "
      f"{RESULTS_DIR}/benchmark_summary.json")
print(f"Accuracy: {RESULTS_DIR}/<model>_eval_results.txt")

In [ ]:
from google.colab import runtime

runtime.unassign()